# Unitree G1 — Flat Terrain Training
Train con robot đi trên mặt phẳng bằng MuJoCo Playground + Brax PPO trên Kaggle Dual T4.

In [ ]:
# ============================================================
# Cell 1: Kiểm tra GPU
# ============================================================
import subprocess

subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"])

In [ ]:
# ============================================================
# Cell 2: Clone repo & cd vào project
# ============================================================
import os, subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/Bipedal-Locomotion")

if not PROJECT_DIR.exists():
    subprocess.check_call([
        "git", "clone",
        "--branch", "M2",
        "https://github.com/TotallyNotMinh/Bipedal-Locomotion.git",
        str(PROJECT_DIR)
    ])
    print("[✓] Cloned repo")
else:
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "pull"])
    print("[✓] Repo up to date")

os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# ============================================================
# Cell 3: Cài đặt dependencies
# ============================================================
import sys, subprocess

pkgs = [
    "brax",
    "flax",
    "optax",
    "orbax-checkpoint",
    "ml_collections",
    "pyyaml",
    "imageio",
    "imageio-ffmpeg",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade"] + pkgs)

# MuJoCo Playground (v0.2.0 có G1JoystickFlatTerrain)
try:
    import mujoco_playground
    print("[✓] mujoco_playground already installed")
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade", "--no-deps",
        "git+https://github.com/google-deepmind/mujoco_playground.git@v0.2.0"
    ])
    print("[✓] mujoco_playground installed")

In [ ]:
# ============================================================
# Cell 4: Kiểm tra JAX & import
# ============================================================
import os

# Phải set TRƯỚC KHI import mujoco để EGL được dùng cho rendering
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import jax
devices = jax.devices()
print(f"[✓] JAX {jax.__version__} — {len(devices)} device(s)")
for d in devices:
    print(f"    {d}")

import mujoco_playground
print(f"[✓] mujoco_playground {getattr(mujoco_playground, '__version__', '?')} imported OK")
print(f"[✓] MUJOCO_GL={os.environ.get('MUJOCO_GL','(not set)')}")


In [ ]:
# ============================================================
# Cell 4b: SAO LƯU / KHÔI PHỤC kết quả (chống mất khi disconnect)
# Chạy trong kernel Kaggle → vẫn backup khi VS Code mất kết nối.
# Lần đầu sẽ hỏi KAGGLE_USERNAME / KAGGLE_KEY (trong file kaggle.json,
#   kaggle.com → Settings → API → Create New Token). Key chỉ nằm trong RAM,
#   không lưu vào notebook. ĐỪNG dán key trực tiếp vào code.
# Kết quả được đẩy lên Kaggle Dataset PRIVATE "g1-m2-results".
# ============================================================
import importlib.util, os, sys

PROJECT_DIR = "/kaggle/working/Bipedal-Locomotion"
os.chdir(PROJECT_DIR)
_spec = importlib.util.spec_from_file_location("backup_results", f"{PROJECT_DIR}/scripts/backup_results.py")
backup_results = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(backup_results)

RESTORE_PREVIOUS = False   # True = tải bản backup cũ về results/flat (khi train tiếp sau disconnect)
if RESTORE_PREVIOUS:
    backup_results.restore(dest_parent="results")

# Tự động backup mỗi 15 phút trong lúc Cell 5 train (thread nền)
backup_results.start_auto_backup(interval_min=15)

In [ ]:
# ============================================================
# Cell 5: TRAIN M2 (capacity-matched MLP) trên mặt phẳng
# Env: default G1JoystickFlatTerrain của Playground (không override)
# PPO: hyperparameter chính thức của Playground cho G1 (200M steps)
# ============================================================
import importlib.util
import os
import sys
import yaml

PROJECT_DIR = "/kaggle/working/Bipedal-Locomotion"
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

spec = importlib.util.spec_from_file_location("train_flat", f"{PROJECT_DIR}/scripts/train_flat.py")
train_flat = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_flat)

with open("configs/ppo_flat.yaml") as f:
    cfg = yaml.safe_load(f)

# Train tiếp từ checkpoint (sau khi khôi phục ở Cell 4b). None = train từ đầu.
# Ví dụ: RESUME_FROM = "results/flat/g1_flat_m2_20261125_120000/checkpoints"
RESUME_FROM = None
if RESUME_FROM:
    cfg["ppo"]["restore_checkpoint_path"] = RESUME_FROM

print("Training timesteps:", f"{cfg['ppo']['num_timesteps']:,}")
print("Model:", cfg["experiment"].get("model_id"), "| policy:", cfg["network"]["policy_hidden_layer_sizes"])
meta = train_flat.train(cfg, output_dir=cfg["experiment"].get("output_dir", "results/flat"))
print("\n=== DONE ===")
print("Checkpoint:", meta["checkpoint_dir"])
print("Total time:", meta["total_wall_time_s"], "s")

# Backup ngay khi train xong
backup_results.backup_now(message="final: training done")

In [ ]:
# ============================================================
# Cell 6 (tuỳ chọn): Xem metrics sau khi train xong
# ============================================================
import glob, json

run_dirs = sorted(glob.glob("results/flat/g1_flat_*"))
if not run_dirs:
    print("Chưa có kết quả train")
else:
    latest = run_dirs[-1]
    rows = []
    with open(f"{latest}/metrics.jsonl") as f:
        for line in f:
            rows.append(json.loads(line))

    print(f"Run: {latest}")
    print(f"{'step':>12}  {'eval_reward':>12}  {'eval_len':>10}  {'wall_time_s':>12}")
    for r in rows:
        print(
            f"{r.get('step', 0):>12,}  "
            f"{r.get('eval/episode_reward', float('nan')):>12.3f}  "
            f"{r.get('eval/avg_episode_length', float('nan')):>10.1f}  "
            f"{r.get('wall_time_s', 0):>12.0f}"
        )

In [ ]:
import glob
import subprocess
import sys
from base64 import b64encode
from pathlib import Path

from IPython.display import HTML

# Tìm checkpoint mới nhất
run_dirs = sorted(glob.glob("results/flat/g1_flat_*"))
assert run_dirs, "Chưa có checkpoint!"
latest_run = Path(run_dirs[-1]).resolve()
ckpt_root = latest_run / "checkpoints"
step_dirs = sorted(
    [d for d in ckpt_root.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name),
)
ckpt_path = str((step_dirs[-1] if step_dirs else ckpt_root).resolve())
video_path = "/kaggle/working/g1_flat_demo.mp4"

print(f"[+] Checkpoint: {ckpt_path}")
print("[+] Chạy render_demo.py trong subprocess...")

result = subprocess.run(
    [
        sys.executable, "scripts/render_demo.py",
        "--checkpoint", ckpt_path,
        "--out", video_path,
        "--steps", "500",
        "--command", "0.5", "0.0", "0.0",   # vx vy wz cố định cho demo
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("=== STDERR ===")
    print(result.stderr)
    raise RuntimeError(f"render_demo.py failed (code {result.returncode})")

print(f"[✓] Video: {video_path}")
mp4 = open(video_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML(
    f'<video width="480" height="360" controls autoplay loop>'
    f'<source src="{data_url}" type="video/mp4"></video>'
))


In [ ]:
# ============================================================
# Cell 9: TẢI VỀ MÁY CÁ NHÂN (khi chạy trên VS Code)
# Nhúng zip (checkpoint mới nhất + metrics + config) vào output của cell.
# VS Code lưu output vào file .ipynb trên máy bạn. Sau đó:
#   1) Ctrl+S notebook
#   2) Trên máy bạn: python scripts/extract_from_notebook.py kaggle_remote.ipynb --out downloads/
# (Hoặc bấm thử link bên dưới.) Xong thì "Clear Output" cell này cho nhẹ notebook.
# ============================================================
import base64, glob, zipfile
from pathlib import Path
from IPython.display import HTML, display

ONLY_LATEST_CKPT = True   # False = lấy tất cả checkpoint (file to hơn)

run = Path(sorted(glob.glob("results/flat/g1_flat_*"))[-1])
ckpts = sorted([d for d in (run / "checkpoints").iterdir() if d.is_dir() and d.name.isdigit()],
               key=lambda d: int(d.name))
keep = ckpts[-1:] if ONLY_LATEST_CKPT else ckpts

zip_path = Path(f"/kaggle/working/{run.name}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in run.iterdir():                      # metrics.jsonl, config, metadata
        if f.is_file():
            zf.write(f, f"{run.name}/{f.name}")
    for d in keep:                               # checkpoint
        for f in d.rglob("*"):
            if f.is_file():
                zf.write(f, f"{run.name}/{f.relative_to(run)}")

size_mb = zip_path.stat().st_size / 1e6
print(f"[+] {zip_path.name}: {size_mb:.1f} MB, checkpoint: {[d.name for d in keep]}")
if size_mb > 80:
    print("[!] File lớn, VS Code có thể bị lag. Đặt ONLY_LATEST_CKPT = True.")

b64 = base64.b64encode(zip_path.read_bytes()).decode()
display(HTML(f'<a download="{zip_path.name}" href="data:application/zip;base64,{b64}">⬇ Tải {zip_path.name}</a>'))